In [1]:
from platform import python_version
print(python_version())

3.11.14


In [2]:
import os, sys, yaml
from pathlib import Path
from dotenv import load_dotenv

import numpy as npmtd
import pandas as pd
pd.set_option('display.width', 100)
pd.set_option('max_colwidth', 80)
pd.set_option("display.precision", 3)

import seaborn as sns
sns.set_context("notebook", font_scale=1.4)

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
%matplotlib inline

sys.path.insert(1, '../src/')

ROOT0 = Path("/home/flavio/uv/perturb_agent/")
ROOT_SRC = ROOT0 / "src"

if str(ROOT_SRC) not in sys.path:
    sys.path.append(str(ROOT_SRC))

print("ROOT0:", ROOT0)
print("ROOT_SRC added:", ROOT_SRC)

from libs.Basic import *
from libs.MTD_lib import MTD
from libs.cBioPortal_lib import cBioPortal
from libs.calc_degs_lib import CALC_DEGS
# from libs.dashcyto_lib import DASH_CYTO
from libs.config_lib import Config

from IPython.display import display, HTML
# display(HTML("<style>.container { width:100% !important; }</style>"))
display(HTML("<style>:root { --jp-notebook-max-width: 100% !important; }</style>"))

with open('../params.yml', 'r') as file:
    dic_yml = yaml.safe_load(file)

# print(dic_yml)

ROOT0: /home/flavio/uv/perturb_agent
ROOT_SRC added: /home/flavio/uv/perturb_agent/src


/home/flavio/uv/perturb_agent/.venv/lib/python3.11/site-packages/Bio/__init__.py:138: BiopythonWarning: You may be importing Biopython from inside the source tree. This is bad practice and might lead to downstream issues. In particular, you might encounter ImportErrors due to missing compiled C extensions. We recommend that you try running your code from outside the source tree. If you are outside the source tree then you have a pyproject.toml file in an unexpected directory: /home/flavio/uv/perturb_agent/.venv/lib/python3.11/site-packages
  warnings.warn(
/home/flavio/uv/perturb_agent/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
email = os.getenv('email')

i_project=0

project_list = dic_yml['project_list']
n = len(project_list)
project = project_list[i_project]

s_project_list = dic_yml['s_project_list']
s_project = s_project_list[i_project]
assert n==len(project_list), f"Error project_list: there are {n} projects"

PROG_ID = 'TCGA'
PSI_ID = 'TCGA-BRCA'
PSI_ID = 'TCGA-ACC'
PSI_ID = 'TCGA-CESC'
PSI_ID = 'TCGA-PAAD'

ROOT0_DATA = ROOT0 / "data"
root_colab = ROOT0_DATA / 'colab'
root_project = ROOT0_DATA / PROG_ID

disease = PSI_ID

root_project = create_dir(ROOT0_DATA, s_project)
root_disease = create_dir(root_project, PSI_ID)

CONTEXT_DISESE = 'xxxx'
context_disease = CONTEXT_DISESE

gene_protein = dic_yml['gene_protein']
s_omics = dic_yml['s_omics']

has_age = dic_yml['has_age']
has_gender = dic_yml['has_gender']

exp_normalization = dic_yml['exp_normalization']
normalization = 'quantile_norm' if exp_normalization == True else 'not_normalized'

LFC_cut_inf = dic_yml['LFC_cut_inf']
s_pathw_enrichm_method = dic_yml['s_pathw_enrichm_method']
ptw_min_num_of_degs_cut = dic_yml['ptw_min_num_of_degs_cut']

tolerance_pPMI = dic_yml['tolerance_pPMI']
type_sat_ptw_index = dic_yml['type_sat_ptw_index']
saturation_lfc_param = dic_yml['saturation_lfc_param']

pval_pathway_cutoff = dic_yml['pval_pathway_cutoff']
fdr_pathway_cutoff = dic_yml['fdr_pathway_cutoff']
num_of_genes_cutoff = dic_yml['num_of_genes_cutoff']
enr_db_list = dic_yml['enr_db_list']


case_list = dic_yml['case_list']
dic_case_list = dic_yml['dic_case_list']

std_filename      = dic_yml['std_filename']
std_filename_list = dic_yml['std_filename_list']

min_lfc_modulation = dic_yml['min_lfc_modulation']
num_of_genes_list  = dic_yml['num_of_genes_list']
pPMI_normalized  = dic_yml['pPMI_normalized']

#--- max len for formatting purposes
s_len_case  = dic_yml['s_len_case']

n_sentences = dic_yml['n_sentences']
run_list = dic_yml['run_list']
chosen_model_list = dic_yml['chosen_model_list']
i_dfp_list = dic_yml['i_dfp_list']
chosen_model_sampling = dic_yml['chosen_model_sampling']

fdr_ptw_cutoff_list = np.arange(0.05, 0.80, 0.05)
lfc_list = np.round(np.arange(1.0, -0.01, -.025), 3)
fdr_list = np.arange(0.05, 0.76, .01)

cfg = Config(root0=ROOT0, root_disease=root_disease, disease=disease, case_list=case_list)
case = case_list[0]

n_genes_annot_ptw, n_degs, n_degs_in_ptw, n_degs_not_in_ptw, degs_in_all_ratio = -1,-1,-1,-1,-1

LFC_cut, lfc_FDR_cut, n_degs, n_degs_up, n_degs_dw = cfg.get_best_lfc_cutoff(case, 'not_normalized')

print(f"project '{project}', s_project '{s_project}'")
print(f"G/P LFC cutoffs: lfc={LFC_cut:.3f}; fdr={lfc_FDR_cut:.3f} - LFC_cut_inf={LFC_cut_inf:.3f}")
print(f"Pathway cutoffs: pval={pval_pathway_cutoff:.3f}; fdr={fdr_pathway_cutoff:.3f}; num of genes={num_of_genes_cutoff}")

Best parameter file for LFC does not exist /home/flavio/uv/perturb_agent/data/TCGA/TCGA-PAAD/config/all_lfc_cutoffs_TCGA-PAAD.tsv
project 'TCGA', s_project 'TCGA'
G/P LFC cutoffs: lfc=1.000; fdr=0.050 - LFC_cut_inf=0.400
Pathway cutoffs: pval=0.050; fdr=0.050; num of genes=3


In [4]:
mtd = MTD(disease=disease, gene_protein=gene_protein, s_omics=s_omics, project=project, s_project=s_project, 
          root0=ROOT0, root0_data=ROOT0_DATA, prog_id=PROG_ID, psi_id=PSI_ID,
          case_list=case_list, dic_case_list=dic_case_list, has_age=has_age, has_gender=has_gender, exp_normalization=exp_normalization,
          std_filename=std_filename, std_filename_list=std_filename_list,
          geneset_num=0, ptw_min_num_of_degs_cut=ptw_min_num_of_degs_cut,
          tolerance_pPMI=tolerance_pPMI, s_pathw_enrichm_method=s_pathw_enrichm_method,
          LFC_cut_inf=LFC_cut_inf, fdr_ptw_cutoff_list=fdr_ptw_cutoff_list,
          num_of_genes_list=num_of_genes_list, lfc_list=lfc_list, fdr_list=fdr_list, 
          min_lfc_modulation=min_lfc_modulation, type_sat_ptw_index=type_sat_ptw_index,
          saturation_lfc_param=saturation_lfc_param, enr_db_list=enr_db_list, pPMI_normalized=pPMI_normalized)

print(">>> Roots", mtd.root0, mtd.root_disease)
case = case_list[0]
print(">>>", case)

mtd.cfg.set_default_best_lfc_cutoff(mtd.normalization, LFC_cut=1, lfc_FDR_cut=0.05)
ret, degs, degs_ensembl, dfdegs = mtd.open_case(case, prompt_verbose=False, verbose=False)
# print("\nEcho Parameters:")
# print(mtd.echo_parameters())

>>> Roots /home/flavio/uv/perturb_agent /home/flavio/uv/perturb_agent/data/TCGA/TCGA-PAAD
>>> Tumor


### cBioPortal - no memory restriction to get all data available

In [5]:
cbio = cBioPortal(root0=ROOT0, root0_data=ROOT0_DATA, memory_restriction=False)

### Get all programs

In [ ]:
verbose = False

df_psi = cbio.open_primary_site(verbose=verbose)
df_psi

,prog_id,cbioportal_study_id,gdc_project_id,psi_id,disease_id,disease_cd,primary_site,disease_context
0,TCGA,paad_tcga_pan_can_atlas_2018,TCGA-PAAD,PAAD,pancreatic_adenocarcinoma,PAAD,Pancreas,"TCGA pancreatic adenocarcinoma, PanCancer Atlas"
1,CPTAC3,paad_cptac_2021,CPTAC-3,PAAD,pancreatic_ductal_adenocarcinoma,PAAD,Pancreas,"CPTAC publication cohort, Cell 2021; 140 pancreatic cancers"
2,CPTAC3,pancreas_cptac_gdc,CPTAC-3,PAAD-GDC,pancreatic_cancer,PAAD,Pancreas,Newer cBioPortal cohort generated from GDC/CDA data in 2025
3,TCGA,skcm_tcga_pan_can_atlas_2018,TCGA-SKCM,SKCM,cutaneous_melanoma,SKCM,Skin,"TCGA skin cutaneous melanoma, PanCancer Atlas"
4,TCGA,brca_tcga_pan_can_atlas_2018,TCGA-BRCA,BRCA,breast_invasive_carcinoma,BRCA,Breast,"TCGA breast invasive carcinoma, PanCancer Atlas"
5,CPTAC2,brca_cptac_2020,CPTAC-2,BRCA,breast_cancer,BRCA,Breast,"CPTAC breast cancer publication cohort, Cell 2020"
6,CPTAC2,breast_cptac_gdc,CPTAC-2,BRCA-GDC,breast_cancer gdc,BRCA,Breast,Newer cBioPortal cohort generated from GDC/CDA data in 2025


In [7]:
"; ".join(cbio.prog_list)

'CPTAC2; CPTAC3; TCGA'

In [8]:
PROG_ID = 'TCGA'
psi_id = 'PAAD'
psi_id = 'SKCM'
psi_id = 'BRCA'


PROG_ID = 'CPTAC2'
psi_id = 'PAAD'
psi_id = 'BRCA'
psi_id = 'BRCA-GDC'

cbio.set_program(prog_id=PROG_ID)

### Open primary cites from cbio

In [9]:
verbose=True
cbio.set_program_and_primary_site(prog_id=PROG_ID, psi_id=psi_id, verbose=verbose)


-----------------------------
>> prog_id: CPTAC2
>> psi_id: BRCA-GDC
>> primary_site: Breast
>> disease_id: breast_cancer gdc
>> disease_cd: BRCA

-----------------------------
>> cbioportal_study_id: breast_cptac_gdc
>> gdc_project_id: CPTAC-2

-----------------------------
>> root disease: /home/flavio/uv/perturb_agent/data/CPTAC2/BRCA-GDC
>> root samples: /home/flavio/uv/perturb_agent/data/CPTAC2/BRCA-GDC/samples
>> root lfc: /home/flavio/uv/perturb_agent/data/CPTAC2/BRCA-GDC/lfc
>> root mutations: /home/flavio/uv/perturb_agent/data/CPTAC2/BRCA-GDC/mutations
-----------------------------



,prog_id,cbioportal_study_id,gdc_project_id,psi_id,disease_id,disease_cd,primary_site,disease_context
6,CPTAC2,breast_cptac_gdc,CPTAC-2,BRCA-GDC,breast_cancer gdc,BRCA,Breast,Newer cBioPortal cohort generated from GDC/CDA data in 2025


In [10]:
cbio.root_disease, cbio.filename_demo

(PosixPath('/home/flavio/uv/perturb_agent/data/CPTAC2/BRCA-GDC'),
 PosixPath('/home/flavio/uv/perturb_agent/data/CPTAC2/BRCA-GDC/clinical_and_demographics_for_breast_cptac_gdc.tsv'))

### Run a program

In [13]:
verbose=False
force=False

prog_id_list = ['CPTAC', 'CCLE', 'TARGET', '']
prog_id_list = ['TCGA']
prog_id_list = ['CPTAC2']
prog_id_list = ['TCGA', 'CPTAC2', 'CPTAC3']

for i, prog_id in enumerate(prog_id_list):

    print(f"{i}) prog_id {prog_id}")

    df_cases, df_all_clin_demo, df_all_samples, df_all_mutations = cbio.loop_program_psi_get_cases_samples_mut(prog_id=prog_id, force=force, verbose=verbose)

    print("")

print("\n--------------- end ---------------")


0) prog_id TCGA
	0) PAAD - Pancreas
..

👉 Returned 185 / Total paginated 185
>>> 152 cases
0-50 ..................50-100 .100-150 .150-152 .		0) TCGA_PAAD_Pancreas_subtype_other_tumor_other_tissue_other
>>> 24 cases
0-24 .........		1) TCGA_PAAD_Pancreas_subtype_adenocarcinoma-generic_tumor_adenocarcinoma_tissue_adenocarcinoma-generic
>>> 6 cases
0-6 ...		2) TCGA_PAAD_Pancreas_subtype_neuroendocrine_tumor_neuroendocrine-tumor_tissue_neuroendocrine
>>> 1 cases
0-1 .		3) TCGA_PAAD_Pancreas_subtype_ductal_tumor_other_tissue_ductal
>>> 1 cases
0-1 .		4) TCGA_PAAD_Pancreas_subtype_other_tumor_melanoma_tissue_other
>>> 1 cases
0-1 .		5) TCGA_PAAD_Pancreas_subtype_squamous_tumor_squamous-cell-carcinoma_tissue_squamous
	1) SKCM - Skin
..

👉 Returned 148 / Total paginated 148
>>> 144 cases
0-50 .................50-100 .100-144 .		0) TCGA_SKCM_Skin_subtype_other_tumor_melanoma_tissue_other
>>> 3 cases
0-3 ..		1) TCGA_SKCM_Skin_subtype_other_tumor_other_tissue_other
>>> 1 cases
0-1 .		2) TCGA_SKCM

In [ ]:
cbio.df_all_clin_demo

### Get cases, subtypes and clin_demo tables

In [ ]:
verbose=True
force=True

df_cases, df_subt, df_clin_demo = cbio.get_cases_and_subtypes(batch_size=200, do_filter=True, force=force, verbose=verbose)

In [ ]:
cbio.df_cases.head(3).T

In [ ]:
df_subt

In [ ]:
df_clin_demo.head(3).T

In [ ]:
df_attr_gdc = cbio.check_clinical_attributes(
    "pancreas_cptac_gdc"
)

df_attr_2021 = cbio.check_clinical_attributes(
    "paad_cptac_2021"
)

display(df_attr_gdc)
display(df_attr_2021)

In [ ]:
prog_ids = ['TCGA-KIRP', 'TCGA-THCA', 'CGCI-BLGSP', 'FM-AD', 'TCGA-LAML', 'TCGA-COAD', 
            'CPTAC-2', 'HCMI-CMDC', 'MMRF-COMMPASS', 'RC-PTCL', 'WCDT-MCRPC', 'ALCHEMIST-ALCH', 'MATCH-U', 'APOLLO-LUAD', 'CMI-ASC', 'MATCH-Z1A',
            'CGCI-HTMCP-DLBCL', 'TCGA-MESO', 'TARGET-ALL-P3', 'VAREPOP-APOLLO', 'MATCH-C1', 'MATCH-S2', 'TCGA-PRAD', 'TARGET-CCSK', 'TARGET-AML', 
            'MATCH-Y', 'TCGA-ACC', 'MATCH-R', 'MP2PRT-ALL', 'TARGET-RT', 'MATCH-Z1D', 'CPTAC-3', 'TCGA-LGG', 'TCGA-SARC', 'APOLLO-OV', 'MATCH-B', 
            'TCGA-LUSC', 'BEATAML1.0-COHORT', 'TCGA-ESCA', 'TARGET-ALL-P1', 'TCGA-KICH', 'OHSU-CNL', 'TCGA-LIHC', 'TCGA-TGCT', 'TCGA-OV', 'MATCH-W', 
            'TCGA-GBM', 'EXCEPTIONAL_RESPONDERS-ER', 'TCGA-READ', 'TARGET-OS', 'TRIO-CRU', 'TCGA-THYM', 'TCGA-HNSC', 'TCGA-UCEC', 'TARGET-NBL', 
            'MATCH-S1', 'TARGET-ALL-P2', 'CDDP_EAGLE-1', 'TCGA-SKCM', 'TCGA-PCPG', 'ORGANOID-PANCREATIC', 'MATCH-N', 'CGCI-HTMCP-CC', 'CCG-CUPP',
            'MATCH-I', 'BEATAML1.0-CRENOLANIB', 'CTSP-DLBCL1', 'CGCI-HTMCP-LC', 'TCGA-STAD', 'TCGA-BRCA', 'NCICCR-DLBCL', 'MP2PRT-WT', 'CCDI-MCI',
            'TCGA-PAAD', 'MATCH-Q', 'MATCH-H', 'TCGA-CHOL', 'REBC-THYR', 'MATCH-Z1B', 'CMI-MBC', 'TARGET-WT', 'TCGA-CESC', 'MATCH-P', 'TCGA-UCS', 'CMI-MPC', 
            'TCGA-DLBC', 'TCGA-KIRC', 'TCGA-LUAD', 'MATCH-Z1I', 'TCGA-UVM', 'TCGA-BLCA']

prog_ids.sort()

print("; ".join(prog_ids))

In [ ]:
all_hits = []
from_ = 0
size_ = 200
total = None

import json, requests

filters = {
    "op": "and",
    "content": [
        {
            "op": "in",
            "content": {
                "field": "project.project_id",
                "value": [cbio.gdc_project_id],
            },
        },
        {
            "op": "in",
            "content": {
                "field": "primary_site",
                "value": [cbio.primary_site],
            },
        },
    ],
}

try:
    while True:
        print(".", end="")

        params = {
            "filters": json.dumps(filters),
            "fields": ",".join(
                [
                    "case_id",
                    "submitter_id",
                    "project.project_id",
                    "primary_site",
                    "disease_type",
                    "diagnoses.primary_diagnosis",
                ]
            ),
            "format": "JSON",
            "size": size_,
            "from": from_,
        }

        res = requests.get(cbio.url_gdc_cases, params=params)
        response = res.json()

        if "data" not in response.keys():
            print(f"No data found while searching for '{cbio.prog_psi_id}'")
            print(">>> response", response)
            raise ValueError("No data found")

        hits = response.get("data", {}).get("hits", [])

        if total is None:
            total = response["data"]["pagination"]["total"]

        if not hits:
            break

        all_hits.extend(hits)
        from_ += size_

    print("\n")

    if all_hits == []:
        print(f"No subtypes found for {cbio.prog_psi_id} - filter value: {cbio.gdc_project_id}")
        cbio.save_case_files(verbose=verbose)
        raise ValueError("No subtypes found")

    # ------------ lost data? ------------------
    N = len(all_hits)

    if N < total:
        print(
            f"⚠️ Warning: results truncated — consider pagination - all hits = {N};  Total paginated {total} "
        )
    else:
        print(f"👉 Returned {N} / Total paginated {total}")

    # ------------ having all hits -------------

    df_cases = pd.json_normalize(all_hits)

    df_cases = df_cases.rename(
        columns={
            "submitter_id": "barcode_case",
            "project.project_id": "gdc_project_id",
        }
    )

except Exception as e:
    print(f"Error for searching diags for '{cbio.prog_psi_id}'. error: {e}")
